# wavexplain — play with promotions

Turn a promotion **on** or **off** and watch the forecast, and the model's **promotion driver**, respond. Turning the promotion channel on or off is an exact counterfactual on the model's input.

Public competition data (Corporacion Favorita). The model's response to the data, not a real-world cause.


## Setup

In [ ]:
# --- Install ---
%pip -q install wavexplain torch numpy pandas matplotlib

# --- CONFIG: point these at your hosted checkpoint + a small sample panel ---
# The demo needs a trained checkpoint and a small slice of the panel the model
# was trained on (series the model has embeddings for). Host a few hundred
# series, not the full dataset.
CKPT_URL   = "https://github.com/kesjien/wavexplain/releases/download/demo/wavenet_demo.pt"   # TODO
DATA_URL   = "https://github.com/kesjien/wavexplain/releases/download/demo/sample_panel.npz"  # TODO
INPUT_LENGTH   = 90   # TODO: the window length your model expects
HORIZON        = 16    # TODO: match your trained model's horizon
NUM_SERIES     = 174685 # TODO: match training
NUM_COVARIATES = 1    # channel 1 = onpromotion

!wget -q -O wavenet_demo.pt "$CKPT_URL"   || echo "set CKPT_URL"
!wget -q -O sample_panel.npz "$DATA_URL"  || echo "set DATA_URL" 

# This is just execution statistics tracked via Google Analytics
from google.colab import userdata
import requests
import uuid

def track_notebook_execution():
    measurement_id = "G-98HEZ541RF"
    
    try:
        api_secret = userdata.get('WaveXplain_Colab')
    except Exception:
        api_secret = None
        
    if not api_secret:
        return

    client_id = str(uuid.uuid4())
    url = f"https://www.google-analytics.com/mp/collect?measurement_id={measurement_id}&api_secret={api_secret}"
    
    payload = {
        "client_id": client_id,
        "events": [{
            "name": "notebook_executed",
            "params": {"notebook_name": "run_on_your_data"}
        }]
    }
    
    try:
        requests.post(url, json=payload, timeout=5)
    except Exception:
        pass

track_notebook_execution()

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from collections import OrderedDict
from wavexplain import MultiSeriesWaveNet, CounterfactualExplainer, render_card_html
from IPython.display import HTML, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the model (constructor args must match training)
model = MultiSeriesWaveNet(num_series=NUM_SERIES, horizon=HORIZON, num_covariates=NUM_COVARIATES)
model.load_state_dict(torch.load("wavenet_demo.pt", map_location=device))  # TODO: adapt if your ckpt is a dict
model.to(device).eval()

# Load a small sample panel: channel 0 = sales, channel 1 = onpromotion
z = np.load("sample_panel.npz")
sales_panel, promo_panel = z["sales_panel"], z["promo_panel"]   # (num_series, num_days)
num_days = sales_panel.shape[1]
start = num_days - INPUT_LENGTH - HORIZON
print("Loaded", sales_panel.shape[0], "series x", num_days, "days")

In [ ]:
def window(series_id):
    """Return the (log-sales, promo) input window for one series."""
    sales = sales_panel[series_id, start:start+INPUT_LENGTH].astype(float)
    promo = promo_panel[series_id, start:start+INPUT_LENGTH].astype(float)
    log_sales = np.log1p(np.clip(sales, 0, None))
    return np.stack([log_sales, promo]), sales, promo

def build_groups(promo, recent_k=14):
    """Split the input timesteps into typical / recent-trend / promotion groups."""
    T = len(promo)
    promo_mask  = promo > 0.5
    recent_mask = np.zeros(T, bool); recent_mask[-recent_k:] = True
    recent_mask &= ~promo_mask
    seasonal_mask = ~(promo_mask | recent_mask)
    return OrderedDict([
        ("seasonal_pattern", seasonal_mask),
        ("recent_trend",     recent_mask),
        ("promotion_effect", promo_mask),
    ])

def explain(series_id, full_input, promo):
    ex = CounterfactualExplainer(model, series_id=series_id, device=device, output_transform=torch.expm1)
    groups = build_groups(promo)
    contributions, baseline_pred, full_pred = ex.explain(
        full_input, baseline_values=[0.0]*(1+NUM_COVARIATES), reveal_groups=groups
    )
    return contributions, baseline_pred, full_pred

## Promotion ON vs OFF

Run the model once with the promotion as it really was, and once with it zeroed, and compare the forecasts.

In [ ]:
def promo_on_off(series_id):
    full_input, sales, promo = window(series_id)
    # ON: promo as-is.  OFF: zero the promotion channel.
    on_input  = full_input.copy()
    off_input = full_input.copy(); off_input[1] = 0.0

    ex = CounterfactualExplainer(model, series_id=series_id, device=device, output_transform=torch.expm1)
    _, _, f_on  = ex.explain(on_input,  baseline_values=[0.0]*(1+NUM_COVARIATES), reveal_groups=build_groups(promo))
    _, _, f_off = ex.explain(off_input, baseline_values=[0.0]*(1+NUM_COVARIATES), reveal_groups=build_groups(np.zeros_like(promo)))

    eff = float(np.sum(f_on) - np.sum(f_off))
    fon, foff = np.atleast_1d(f_on), np.atleast_1d(f_off)
    d = np.arange(len(fon))
    plt.figure(figsize=(9,4))
    plt.plot(d, foff, "o--", color="tab:gray", label="Promotion OFF")
    plt.plot(d, fon,  "o-",  color="tab:cyan", label="Promotion ON")
    plt.title(f"Series {series_id}: promotion effect = {eff:+.1f} units over the horizon")
    plt.xlabel("Forecast day"); plt.ylabel("Predicted units"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    return eff

_ = promo_on_off(0)

## Inject your own promotion

Set the promotion flag on for the last few days of history and see the forecast move.

In [ ]:
def inject_promo(series_id, last_k=7):
    full_input, sales, promo = window(series_id)
    injected = promo.copy(); injected[-last_k:] = 1.0

    base_input = full_input.copy()
    inj_input  = full_input.copy(); inj_input[1] = injected

    ex = CounterfactualExplainer(model, series_id=series_id, device=device, output_transform=torch.expm1)
    _, _, f_base = ex.explain(base_input, baseline_values=[0.0]*(1+NUM_COVARIATES), reveal_groups=build_groups(promo))
    _, _, f_inj  = ex.explain(inj_input,  baseline_values=[0.0]*(1+NUM_COVARIATES), reveal_groups=build_groups(injected))

    lift = float(np.sum(f_inj) - np.sum(f_base))
    fb, fi = np.atleast_1d(f_base), np.atleast_1d(f_inj); d = np.arange(len(fb))
    plt.figure(figsize=(9,4))
    plt.plot(d, fb, "o--", color="tab:gray",   label="Original")
    plt.plot(d, fi, "o-",  color="tab:orange", label=f"Promo injected (last {last_k}d)")
    plt.title(f"Series {series_id}: injecting a promotion moves the forecast by {lift:+.1f} units")
    plt.xlabel("Forecast day"); plt.ylabel("Predicted units"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    return lift

_ = inject_promo(0, last_k=7)

## The bigger picture: promotion effect across many series

One example proves nothing. Run the promotion counterfactual across a sample and show the distribution.

In [ ]:
def promo_distribution(n_series=40):
    effs = []
    for sid in range(min(n_series, sales_panel.shape[0])):
        _, _, promo = window(sid)
        if promo.sum() == 0:  # only series that had promotions
            continue
        full_input, _, promo = window(sid)
        on_input  = full_input.copy()
        off_input = full_input.copy(); off_input[1] = 0.0
        ex = CounterfactualExplainer(model, series_id=sid, device=device, output_transform=torch.expm1)
        _, _, f_on  = ex.explain(on_input,  baseline_values=[0.0]*(1+NUM_COVARIATES), reveal_groups=build_groups(promo))
        _, _, f_off = ex.explain(off_input, baseline_values=[0.0]*(1+NUM_COVARIATES), reveal_groups=build_groups(np.zeros_like(promo)))
        effs.append(float(np.sum(f_on) - np.sum(f_off)))
    effs = np.array(effs)
    print(f"Series with promotions: {len(effs)}")
    print(f"Mean {effs.mean():+.1f} | median {np.median(effs):+.1f} | share positive {(effs>0).mean()*100:.0f}%")
    plt.figure(figsize=(9,4))
    plt.hist(effs, bins=20, color="tab:cyan", edgecolor="white", alpha=0.85)
    plt.axvline(0, color="tab:gray", lw=1)
    plt.title(f"Promotion effect across {len(effs)} series")
    plt.xlabel("Effect on forecast (units)"); plt.ylabel("Series"); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    return effs

_ = promo_distribution(40)

## Notes

- Turning the promotion channel on/off is an exact counterfactual on the model's input; it shows the learned association, not a proven real-world causal effect.
- Runs on trained series only; explaining new data requires fitting a model first.
- For the faithfulness of the attribution itself (deletion/insertion), see `faithfulness_test.py`.
